In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, make_scorer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import cross_val_score, train_test_split
import xgboost as xgb
from sklearn.feature_selection import SelectFromModel
import joblib

Data Prep

In [2]:
data_path = r"ml_training_data.csv"
try:
    df = pd.read_csv(data_path)
    if df.empty or df.isnull().sum().sum() > 0:
        print("Warning: Data contains missing values or is empty.")
except FileNotFoundError:
    raise FileNotFoundError(f"File not found at {data_path}")

In [3]:
def engineer_features(df):
    """
    Create engineered features from the original dataset
    """
    df = df.copy()
    
    # Calculate derived features
    df['temp_rainfall_ratio'] = df['temperature'] / (df['rainfall'] + 1)  # Add 1 to avoid division by zero
    df['npk_total'] = df['N_level'] + df['P_level'] + df['K_level']
    df['temp_squared'] = df['temperature'] ** 2
    
    # Create categorical features
    df['temp_category'] = pd.qcut(df['temperature'], q=4, 
                                labels=['Low', 'Medium', 'High', 'Very High'])
    df['rainfall_category'] = pd.qcut(df['rainfall'], q=4, 
                                    labels=['Low', 'Medium', 'High', 'Very High'])
    
    return df

In [4]:
data = engineer_features(df)

In [5]:
print("Dataset Shape:", data.shape)
display(data.head())
data.info()

Dataset Shape: (500, 16)


,N_level,P_level,K_level,pH,rainfall,temperature,growth_stage,N_rec,P_rec,K_rec,pH_adj,temp_rainfall_ratio,npk_total,temp_squared,temp_category,rainfall_category
0,183.91,37.53,117.34,6.32,39.59,17.44,Mature,86,50,151,Apply Lime,0.429662,338.78,304.1536,Medium,Low
1,183.57,21.61,144.46,5.50,243.88,28.69,Mature,131,46,137,Apply Lime,0.117159,349.64,823.1161,Very High,Very High
2,225.33,45.80,235.63,5.97,180.22,31.05,Mature,55,22,181,Use Ammonium Sulfate,0.171339,506.76,964.1025,Very High,High
3,26.95,0.39,1.85,5.08,273.03,24.06,Mature,129,26,65,Apply Lime,0.087801,29.19,578.8836,High,Very High
4,19.37,20.49,219.95,4.96,273.66,27.10,Young,116,41,72,No Adjustment,0.098667,259.81,734.4100,High,Very High


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   N_level              500 non-null    float64 
 1   P_level              500 non-null    float64 
 2   K_level              500 non-null    float64 
 3   pH                   500 non-null    float64 
 4   rainfall             500 non-null    float64 
 5   temperature          500 non-null    float64 
 6   growth_stage         500 non-null    object  
 7   N_rec                500 non-null    int64   
 8   P_rec                500 non-null    int64   
 9   K_rec                500 non-null    int64   
 10  pH_adj               500 non-null    object  
 11  temp_rainfall_ratio  500 non-null    float64 
 12  npk_total            500 non-null    float64 
 13  temp_squared         500 non-null    float64 
 14  temp_category        500 non-null    category
 15  rainfall_category    50

In [6]:
def create_preprocessing_pipeline():
    """
    Create a preprocessing pipeline for feature transformation
    """
    numeric_features = ['N_level', 'P_level', 'K_level', 'pH', 'rainfall', 'temperature', 
                       'temp_rainfall_ratio', 'npk_total', 'temp_squared']
    categorical_features = ['growth_stage', 'temp_category', 'rainfall_category']
    
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])
    
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(drop='first', sparse_output=False))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])
    
    return preprocessor

In [7]:
class AgriculturalModel:
    def __init__(self):
        self.npk_model = None
        self.ph_model = None
        self.preprocessor = None
        self.feature_selector = None
    
    def fit(self, X, y_npk, y_ph):
        # Create and fit preprocessing pipeline
        self.preprocessor = create_preprocessing_pipeline()
        X_processed = self.preprocessor.fit_transform(X)
        
        # Feature selection for NPK prediction
        self.feature_selector = SelectFromModel(
            GradientBoostingRegressor(random_state=42),
            max_features=15
        )
        X_selected = self.feature_selector.fit_transform(X_processed, y_npk['N_rec'])
        
        # Train NPK model with XGBoost
        npk_model = MultiOutputRegressor(
            xgb.XGBRegressor(
                objective='reg:squarederror',
                n_estimators=300,
                max_depth=8,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42
            )
        )
        
        # Train pH model with RandomForest
        ph_model = RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight='balanced',
            random_state=42
        )
        
        self.npk_model = npk_model.fit(X_selected, y_npk)
        self.ph_model = ph_model.fit(X_processed, y_ph)
        
    def predict(self, X):
        X_processed = self.preprocessor.transform(X)
        X_selected = self.feature_selector.transform(X_processed)
        
        npk_pred = self.npk_model.predict(X_selected)
        ph_pred = self.ph_model.predict(X_processed)
        
        return npk_pred, ph_pred

In [8]:
feature_columns = ['N_level', 'P_level', 'K_level', 'pH', 'rainfall', 'temperature', 
                  'growth_stage', 'temp_rainfall_ratio', 'npk_total', 'temp_squared',
                  'temp_category', 'rainfall_category']


In [9]:
X = data[feature_columns]
y_npk = data[['N_rec', 'P_rec', 'K_rec']]
y_pH = data['pH_adj']

In [10]:
X_train, X_test, y_train_npk, y_test_npk = train_test_split(X, y_npk, test_size=0.2, random_state=42)
_, _, y_train_pH, y_test_pH = train_test_split(X, y_pH, test_size=0.2, random_state=42)


In [11]:
model = AgriculturalModel()
model.fit(X_train, y_train_npk, y_train_pH)
model_save_path=f"agricultural_model.sav"
joblib.dump(model, model_save_path)

['agricultural_model.sav']

In [12]:
npk_pred, ph_pred = model.predict(X_test)

In [13]:
mse_values = [
    mean_squared_error(y_test_npk.iloc[:, i], npk_pred[:, i]) 
    for i in range(y_test_npk.shape[1])
]

print("\nNPK Prediction Results:")
for i, col in enumerate(y_test_npk.columns):
    print(f"{col} MSE: {mse_values[i]:.4f}")
print(f"Overall NPK MSE: {np.mean(mse_values):.4f}")

print("\npH Classification Results:")
print(f"Accuracy: {accuracy_score(y_test_pH, ph_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_pH, ph_pred))


NPK Prediction Results:
N_rec MSE: 2522.3281
P_rec MSE: 93.0864
K_rec MSE: 2319.0391
Overall NPK MSE: 1644.8179

pH Classification Results:
Accuracy: 0.3600

Classification Report:
                      precision    recall  f1-score   support

          Apply Lime       0.33      0.42      0.37        31
       No Adjustment       0.40      0.39      0.39        36
Use Ammonium Sulfate       0.35      0.27      0.31        33

            accuracy                           0.36       100
           macro avg       0.36      0.36      0.36       100
        weighted avg       0.36      0.36      0.36       100



In [14]:
new_data = pd.DataFrame({
    'N_level': [120],
    'P_level': [30],
    'K_level': [100],
    'pH': [5.5],
    'rainfall': [180],
    'temperature': [25],
    'growth_stage': ['Mature'],
    'temp_rainfall_ratio': [25/180],
    'npk_total': [250],
    'temp_squared': [625],
    'temp_category': ['Medium'],
    'rainfall_category': ['High']
})


In [15]:
npk_predictions, ph_predictions = model.predict(new_data)

In [16]:
print("Predictions for new data:")
print(f"NPK Recommendations (kg/ha):")
print(f"N: {npk_predictions[0][0]:.2f}")
print(f"P: {npk_predictions[0][1]:.2f}")
print(f"K: {npk_predictions[0][2]:.2f}")
print(f"\npH Adjustment Recommendation: {ph_predictions[0]}")

Predictions for new data:
NPK Recommendations (kg/ha):
N: 159.86
P: 36.92
K: 108.20

pH Adjustment Recommendation: No Adjustment
